# Stage 10 — risk-policy fit on FoR val-B (risk-v1)

This notebook runs the official Stage 10 risk-policy fit (Protocol v1.0 §6; D2, D9, D44, D45) from an exact clean commit: it builds the val-A and val-B log-Mel caches twice and compares their bytes, copies the Stage 8 bundle-seed checkpoint (seed 0) from Drive, runs `src.tuning.risk_run` once and exports the policy, the per-clip val-B scores and the reports to Drive. The runner checks the checkpoint against the committed Stage 8 evidence, reproduces its val-A EER, then scores val-B and fits the policy ([docs/risk_run.md](../docs/risk_run.md)). Only val-A and val-B are read: train and test are never cached, and the test split is never requested (D31). Scoring runs on CPU by design, so no GPU runtime is needed. Nothing is logged to an external service; the committed reports are the record (D40). Run it only after the implementation commit is reviewed and merged.

## 1. Parameters

Replace `COMMIT` with the full 40-character approved commit. `VERSION` names the risk-policy folder and is passed to the runner. `CNN_VERSION` names the Stage 8 checkpoint folder the bundle-seed checkpoint is copied from. There are no other knobs: the scoring batch size comes from `configs/cnn.yaml` and the policy fit from `src.tuning.risk_fit`.

In [ ]:
REPO_URL = 'https://github.com/kahlanzzi/voice-spoofing-detection.git'
COMMIT = 'REPLACE_WITH_APPROVED_COMMIT'  # full 40-character SHA
VERSION = 'risk-v1'
CNN_VERSION = 'cnn-v1'
DRIVE_ROOT = '/content/drive/MyDrive/voice-spoofing-detection'

## 2. Mount Drive and refuse existing outputs

Drive is mounted before anything else. A frozen policy or evidence folder is never overwritten, so the notebook stops here, before any expensive work, if `checkpoints/risk/<VERSION>/` or `processed/metadata/stage10/` already exists. The same check runs again right before export.

In [ ]:
from pathlib import Path
import hashlib, json, os, shutil, subprocess, sys

from google.colab import drive
drive.mount('/content/drive')

DRIVE = Path(DRIVE_ROOT)
CHECKPOINT_DEST = DRIVE / 'checkpoints' / 'risk' / VERSION
EVIDENCE_DEST = DRIVE / 'processed' / 'metadata' / 'stage10'


def refuse_existing_outputs():
    for dest in (CHECKPOINT_DEST, EVIDENCE_DEST):
        if dest.exists():
            raise FileExistsError(f'{dest} already exists; Stage 10 outputs are never overwritten')


def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()


refuse_existing_outputs()
assert (DRIVE / 'raw').is_dir(), f'{DRIVE / "raw"} not found; check DRIVE_ROOT'

## 3. Check out the exact commit

The repository is cloned into `/content` and the full commit is checked out detached. HEAD must equal `COMMIT` exactly and the worktree must be clean. Every later output (caches, pytest XML, the checkpoint copy, the run output) is written outside the clone, so the runner, which refuses a dirty worktree, sees a clean tree. `assert_clean` also refuses ignored files other than Python bytecode caches, because `.gitignore` hides some folders from `git status`.

In [ ]:
REPO = Path('/content/voice-spoofing-detection')
assert len(COMMIT) == 40, f'COMMIT must be the full 40-character SHA, got {COMMIT!r}'
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(['git', 'clone', REPO_URL, str(REPO)], check=True)
subprocess.run(['git', 'checkout', '--detach', COMMIT], cwd=REPO, check=True)


def git(*args):
    return subprocess.run(['git', *args], cwd=REPO, check=True, capture_output=True, text=True).stdout


def assert_clean():
    head = git('rev-parse', 'HEAD').strip()
    assert head == COMMIT, (head, COMMIT)
    status = git('status', '--porcelain')
    assert not status, status
    ignored = [line[3:] for line in git('status', '--porcelain', '--ignored').splitlines() if line.startswith('!! ')]
    unexpected = [path for path in ignored if not path.rstrip('/').endswith('__pycache__')]
    assert not unexpected, unexpected


assert_clean()
os.chdir(REPO)
print('HEAD', COMMIT)

## 4. Environment: Colab's torch, then the pinned requirements

The runtime must be Python 3.13 (D15). Colab's preinstalled torch is used and must be 2.11.0 (`requirements-cnn.txt`, D42). Torch is never installed by this notebook: only `requirements.txt` and `requirements-cnn.txt` are installed, and neither lists torch. The versions are then read again in a fresh process, so a silent upgrade during the install is caught. No GPU is required or checked: the runner scores on CPU by design ([docs/risk_run.md](../docs/risk_run.md) Why (c)), so a CPU runtime is enough.

In [ ]:
import torch

assert torch.__version__.startswith('2.11.0'), torch.__version__
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'requirements.txt', '-r', 'requirements-cnn.txt'],
               cwd=REPO, check=True)
PROBE = '''
import json, platform, numpy, torch
print(json.dumps({
    'python': platform.python_version(),
    'numpy': numpy.__version__,
    'torch': torch.__version__,
}))
'''
ENV = json.loads(subprocess.run([sys.executable, '-c', PROBE], check=True, capture_output=True, text=True).stdout)
assert ENV['torch'] == torch.__version__ and ENV['torch'].startswith('2.11.0'), (ENV['torch'], torch.__version__)
assert ENV['python'].startswith('3.13.'), f"Python 3.13 is required (D15), got {ENV['python']}"
assert_clean()
for key, value in ENV.items():
    print(f'{key}: {value}')

## 5. Verify and extract the pinned FoR archive

As in notebooks 03 and 04, the revision marker next to the Drive backup must equal the D21 FoR revision, read here from `configs/audit.yaml` (the D21 pin location). The archive is extracted with `filter='data'`. Its SHA-256 is computed and written to the export manifest as `for_archive_sha256_observed`: this is evidence only, never asserted or compared, and not a dataset pin (D21 unchanged).

In [ ]:
import tarfile
import yaml

EXPECTED_REVISION = yaml.safe_load((REPO / 'configs/audit.yaml').read_text(encoding='utf-8'))['for2sec']['revision']
ARCHIVE = DRIVE / 'raw' / 'for_2sec_ff8c82c7.tar'
REVISION_FILE = DRIVE / 'raw' / 'for_2sec_REVISION.txt'
assert REVISION_FILE.read_text(encoding='utf-8').strip() == EXPECTED_REVISION
DATA_ROOT = Path('/content/data')
AUDIO_ROOT = DATA_ROOT / 'for_2sec'
LOCAL_ARCHIVE = DATA_ROOT / 'for_2sec_ff8c82c7.tar'
DATA_ROOT.mkdir(parents=True, exist_ok=True)
shutil.copy2(ARCHIVE, LOCAL_ARCHIVE)
if AUDIO_ROOT.exists():
    shutil.rmtree(AUDIO_ROOT)
with tarfile.open(LOCAL_ARCHIVE) as archive:
    archive.extractall(DATA_ROOT, filter='data')
FOR_ARCHIVE_SHA256_OBSERVED = sha256(LOCAL_ARCHIVE)  # recorded only (evidence, not a pin)
print('FoR archive extracted to', AUDIO_ROOT)
print('for_archive_sha256_observed:', FOR_ARCHIVE_SHA256_OBSERVED)

## 6. Run the test suite: no failure, error or skip

`REQUIRE_TORCH_TESTS=1` turns a missing torch into a failure. The counts come from the JUnit XML written to `/content` (outside the clone), not from the terminal text, and every failure, error and skip stops the notebook.

In [ ]:
import xml.etree.ElementTree as ET


def stream(command, **kwargs):
    # Like subprocess.run(check=True), but prints the child's output line by line as it runs.
    env = {**kwargs.pop('env', os.environ), 'PYTHONUNBUFFERED': '1'}
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env,
                          **kwargs) as process:
        for line in process.stdout:
            print(line, end='')
    if process.returncode:
        raise subprocess.CalledProcessError(process.returncode, command)


JUNIT = Path('/content/pytest_stage10.xml')
if JUNIT.exists():
    JUNIT.unlink()
assert_clean()
stream([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', f'--junitxml={JUNIT}'],
       cwd=REPO, env={**os.environ, 'REQUIRE_TORCH_TESTS': '1'})
root = ET.parse(JUNIT).getroot()
suites = [root] if root.tag == 'testsuite' else root.findall('testsuite')
COUNTS = {key: sum(int(suite.get(key, 0)) for suite in suites) for key in ('tests', 'failures', 'errors', 'skipped')}
print(COUNTS)
assert COUNTS['tests'] > 0, COUNTS
assert COUNTS['failures'] == COUNTS['errors'] == COUNTS['skipped'] == 0, COUNTS
assert_clean()

## 7. Build the val-A and val-B log-Mel caches twice and compare the bytes

Each split is cached into `/content/cache_a` and again into `/content/cache_b`; every sample hash is checked before decoding. Both builds must hold the same files with identical SHA-256s, and every cache must record the F3 profile config SHA-256 and fingerprint stored in the repo (Stage 6 evidence, D39). val-B is cached here for the first time: D42 deferred it to Stage 10. Train and test are never built. The runner reads `cache_a` and checks the val-A cache against the `npz_sha256` the checkpoint recorded at Stage 8.

In [ ]:
MANIFEST = REPO / 'data/manifests/for2sec_frozen_manifest.csv'
PROFILE = REPO / 'configs/preprocessing.yaml'
F3 = json.loads((REPO / 'docs/audit_evidence/stage6_preprocessing/profile_report.json').read_text(encoding='utf-8'))['profile']
assert sha256(PROFILE) == F3['config_sha256'], 'configs/preprocessing.yaml differs from F3'
CACHE_SPLITS = ('val-A', 'val-B')
CACHE_A, CACHE_B = Path('/content/cache_a'), Path('/content/cache_b')
for cache in (CACHE_A, CACHE_B):
    if cache.exists():
        shutil.rmtree(cache)
    cache.mkdir()
    for split in CACHE_SPLITS:
        assert_clean()
        subprocess.run([
            sys.executable, '-m', 'src.features.logmel_cache',
            '--root', str(AUDIO_ROOT), '--manifest', str(MANIFEST), '--profile', str(PROFILE),
            '--split', split, '--out', str(cache / split),
        ], cwd=REPO, check=True, stdout=subprocess.DEVNULL)


def file_hashes(folder):
    return {p.relative_to(folder).as_posix(): sha256(p) for p in sorted(folder.rglob('*')) if p.is_file()}


CACHE_HASHES_A, CACHE_HASHES_B = file_hashes(CACHE_A), file_hashes(CACHE_B)
assert sorted(CACHE_HASHES_A) == sorted(f'{s}/{s}.{ext}' for s in CACHE_SPLITS for ext in ('npz', 'json')), sorted(CACHE_HASHES_A)
assert CACHE_HASHES_B == CACHE_HASHES_A, 'the two cache builds differ'
for split in CACHE_SPLITS:
    meta = json.loads((CACHE_A / split / f'{split}.json').read_text(encoding='utf-8'))
    assert meta['profile_config_sha256'] == F3['config_sha256'], (split, meta['profile_config_sha256'])
    assert meta['profile_fingerprint'] == F3['fingerprint'], (split, meta['profile_fingerprint'])
    assert meta['code_commit'] == COMMIT, (split, meta['code_commit'])
assert_clean()
CACHE_HASHES_A

## 8. Copy the bundle-seed checkpoint from Drive

The Stage 8 bundle-seed checkpoint, `checkpoints/cnn/<CNN_VERSION>/seed_0/` (D45: seed 0 only), is copied to `/content/stage8_checkpoint/seed_0`, outside the clone. It is not verified or loaded here: the runner checks its bytes against `checkpoint_sha256` and its seed against `bundle_seed` in the committed Stage 8 evidence (`docs/audit_evidence/stage8_cnn/`), before any cache is read.

In [ ]:
CNN_CHECKPOINT = DRIVE / 'checkpoints' / 'cnn' / CNN_VERSION / 'seed_0'
CHECKPOINT_LOCAL = Path('/content/stage8_checkpoint') / 'seed_0'
assert CNN_CHECKPOINT.is_dir(), f'{CNN_CHECKPOINT} not found'
if CHECKPOINT_LOCAL.parent.exists():
    shutil.rmtree(CHECKPOINT_LOCAL.parent)
CHECKPOINT_LOCAL.parent.mkdir()
shutil.copytree(CNN_CHECKPOINT, CHECKPOINT_LOCAL)
assert not CHECKPOINT_LOCAL.resolve().is_relative_to(REPO.resolve()), CHECKPOINT_LOCAL
print('copied', CNN_CHECKPOINT, '->', CHECKPOINT_LOCAL)
print(sorted(p.name for p in CHECKPOINT_LOCAL.iterdir()))

## 9. Run the risk-policy fit once

One call to `src.tuning.risk_run` checks the copied checkpoint against the Stage 8 evidence, runs the provenance checks, reproduces the recorded val-A EER on CPU, scores val-B twice (identical), fits the policy and refits it (byte-identical), and writes four files into `/content/stage10_out`, outside the clone. That folder must not exist yet; on any failure the runner removes it.

Rerun rule (pre-declared 2026-10-28): if the runner raises a check failure (checkpoint hash, seed, provenance, val-A reproduction, completeness, determinism or fit), Stage 10 stops and the failure is reported; proceeding requires a logged decision. Only an infrastructure failure that happens before this cell starts (Drive, clone, pip, archive) may be rerun from the same COMMIT. Nothing in this notebook reads or reacts to the fitted values.

In [ ]:
OUT = Path('/content/stage10_out')
if OUT.exists():
    raise FileExistsError(f'{OUT} already exists; the risk run runs once per runtime')
assert_clean()
stream([
    sys.executable, '-m', 'src.tuning.risk_run',
    '--cache-dir', str(CACHE_A), '--checkpoint', str(CHECKPOINT_LOCAL), '--profile', str(PROFILE),
    '--config', str(REPO / 'configs/cnn.yaml'), '--manifest', str(MANIFEST),
    '--stage8-evidence', str(REPO / 'docs/audit_evidence/stage8_cnn'), '--out', str(OUT), '--version', VERSION,
], cwd=REPO)

## 10. Post-run checks

The worktree must still be clean at `COMMIT`, which shows that nothing wrote into the clone. `run_report.json` must record `COMMIT`, the F3 profile fingerprint and config SHA-256, and bundle seed 0 (D45). `diagnostics_report.json` is printed for review only: it is descriptive, and no change to the calibration, threshold or band follows from it ([docs/risk_run.md](../docs/risk_run.md) Why (e)).

In [ ]:
assert_clean()
RUN_REPORT = json.loads((OUT / 'run_report.json').read_text(encoding='utf-8'))
assert RUN_REPORT['code_commit'] == COMMIT, RUN_REPORT['code_commit']
assert RUN_REPORT['profile_fingerprint'] == F3['fingerprint'], RUN_REPORT['profile_fingerprint']
assert RUN_REPORT['profile_config_sha256'] == F3['config_sha256'], RUN_REPORT['profile_config_sha256']
assert RUN_REPORT['bundle_seed'] == 0, RUN_REPORT['bundle_seed']
print((OUT / 'diagnostics_report.json').read_text(encoding='utf-8'))

## 11. Export to Drive and write the export manifest

`risk_policy.json` and `val_b_scores.npz` go to `checkpoints/risk/<VERSION>/`, and `run_report.json` and `diagnostics_report.json` to `processed/metadata/stage10/`. Both folders are re-checked and created with `exist_ok=False`, so nothing is overwritten. Every Drive copy is re-hashed against its `/content` source, and `stage10_export_manifest.json` (paths relative to `DRIVE_ROOT`, sizes, SHA-256s, commit, versions, environment, UTC time and the observed FoR archive SHA-256, which is evidence only) is written next to the reports. It also records the D21 FoR revision, the pytest counts, the SHA-256 of every cache file of both builds (checked identical in step 7) and the F3 config SHA-256 and fingerprint the caches were checked against.

Two hash checks run. The first, right after copying, checks the copy. Drive is then flushed, unmounted and remounted, and every file is hashed again against the manifest: this second check covers what Drive actually stored after the flush, and also fails on a missing or extra file. These are candidate records until human review.

In [ ]:
from datetime import datetime, timezone

refuse_existing_outputs()
EXPORTS = {
    'risk_policy.json': CHECKPOINT_DEST,
    'val_b_scores.npz': CHECKPOINT_DEST,
    'run_report.json': EVIDENCE_DEST,
    'diagnostics_report.json': EVIDENCE_DEST,
}
assert sorted(p.name for p in OUT.iterdir()) == sorted(EXPORTS), sorted(p.name for p in OUT.iterdir())
CHECKPOINT_DEST.parent.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DEST.mkdir(exist_ok=False)
EVIDENCE_DEST.parent.mkdir(parents=True, exist_ok=True)
EVIDENCE_DEST.mkdir(exist_ok=False)
files = []
for name, dest in EXPORTS.items():
    shutil.copy2(OUT / name, dest / name)
    digest = sha256(OUT / name)
    assert sha256(dest / name) == digest, f'{dest / name} differs from {OUT / name}'
    files.append({'path': (dest / name).relative_to(DRIVE).as_posix(), 'size': (dest / name).stat().st_size,
                  'sha256': digest})
EXPORT_MANIFEST = {
    'commit': COMMIT,
    'version': VERSION,
    'cnn_version': CNN_VERSION,
    'python': ENV['python'],
    'numpy': ENV['numpy'],
    'torch': ENV['torch'],
    'created_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'for_revision': EXPECTED_REVISION,
    'for_archive_sha256_observed': FOR_ARCHIVE_SHA256_OBSERVED,
    'pytest_counts': COUNTS,
    'cache_sha256': {'cache_a': CACHE_HASHES_A, 'cache_b': CACHE_HASHES_B},
    'f3': {'config_sha256': F3['config_sha256'], 'fingerprint': F3['fingerprint']},
    'files': files,
}
MANIFEST_PATH = EVIDENCE_DEST / 'stage10_export_manifest.json'
MANIFEST_TEXT = json.dumps(EXPORT_MANIFEST, indent=2, sort_keys=True) + '\n'
MANIFEST_PATH.write_text(MANIFEST_TEXT, encoding='utf-8')
print(MANIFEST_TEXT, end='')
drive.flush_and_unmount()

## 12. Remount and re-verify every exported file

Drive is remounted and every file under the two export folders is hashed again against the manifest (the manifest itself against the text just written). A missing file, an extra file or a changed byte stops the notebook.

In [ ]:
drive.mount('/content/drive', force_remount=True)
expected = {entry['path']: entry['sha256'] for entry in EXPORT_MANIFEST['files']}
expected[MANIFEST_PATH.relative_to(DRIVE).as_posix()] = hashlib.sha256(MANIFEST_TEXT.encode('utf-8')).hexdigest()
stored = {p.relative_to(DRIVE).as_posix() for dest in (CHECKPOINT_DEST, EVIDENCE_DEST)
          for p in dest.rglob('*') if p.is_file()}
missing, extra = sorted(set(expected) - stored), sorted(stored - set(expected))
assert not missing and not extra, f'missing on Drive: {missing}; extra on Drive: {extra}'
mismatched = [path for path, digest in sorted(expected.items()) if sha256(DRIVE / path) != digest]
assert not mismatched, f'SHA-256 mismatch on Drive after remount: {mismatched}'
print(f'Drive verified after remount: {len(expected)} files match the manifest.')

## 13. What is committed

After human review, four files are committed under `docs/audit_evidence/stage10_risk/`: `risk_policy.json`, `run_report.json` and `diagnostics_report.json` from this run, and `stage10_export_manifest.json`. `val_b_scores.npz` holds one line per val-B clip; it is data and stays on Drive only, bound to the reports by `val_b_scores_sha256` in `run_report.json` ([docs/risk_run.md](../docs/risk_run.md), "What is committed"; AGENTS.md rule 7). `risk_policy.json` is the policy file Stage 12 freezes into the ModelBundle.